# Maximum Likelihood & Bayesian Inference

## What's covered

- The **likelihood function** — same equation as the PDF/PMF, different interpretation
- **Maximum Likelihood Estimation (MLE)** — find the parameters that make the data most plausible
- MLE for **Bernoulli, Gaussian, and Linear Regression** — three worked examples
- **Bayesian inference** — prior, likelihood, posterior; reasoning about parameters as random variables
- **Maximum A Posteriori (MAP)** — Bayesian point estimate, equivalent to **regularized MLE**
- **Conjugate priors** — Beta-Bernoulli and Gaussian-Gaussian; why they're convenient
- The MLE-to-loss-function bridge — why **MSE = MLE under Gaussian**, **cross-entropy = MLE under Categorical**
- Where this appears in ML — every training run, every loss function


## The likelihood function

You have data `D = \{x_1, \dots, x_n\}` and a model `p(x | \theta)` with parameters `\theta`. The **likelihood** is exactly the same equation as the joint PDF/PMF — but viewed differently:

$$
L(\theta) = p(D | \theta) = \prod_{i=1}^n p(x_i | \theta) \quad \text{(for i.i.d. data)}
$$

**The crucial reinterpretation.**

- `p(x | \theta)`, as a function of `x` with `\theta` fixed, is a **probability** — sums (or integrates) to 1 over `x`.
- `L(\theta)`, as a function of `\theta` with `x` fixed, is a **likelihood** — does NOT sum to 1 over `\theta`. It's not a probability distribution at all. It's just "how plausible would the data have been, if this `\theta` were true?"

The likelihood is the bridge between *data* and *parameter inference*. Everything in this notebook — MLE, MAP, Bayesian inference — manipulates it.

**The log-likelihood.** Products of small numbers underflow fast. We always work with the log:

$$
\ell(\theta) = \log L(\theta) = \sum_{i=1}^n \log p(x_i | \theta)
$$

Sums are friendly to gradients and to floating point. **Negative log-likelihood (NLL)** is what you minimize in nearly every neural network training run. Cross-entropy *is* NLL for categorical outputs. MSE *is* NLL for Gaussian outputs.


## Maximum Likelihood Estimation

MLE picks the `\theta` under which the observed data are most plausible:

$$
\hat{\theta}_{\text{MLE}} = \arg\max_\theta L(\theta) = \arg\max_\theta \ell(\theta) = \arg\min_\theta \bigl[-\ell(\theta)\bigr]
$$

The minimum-of-negative-log form is what training loops actually compute.

**Recipe.**

1. Write the likelihood as a product over data points.
2. Take logs to turn it into a sum.
3. Differentiate with respect to `\theta`, set to zero.
4. Solve for `\theta` analytically, or use gradient descent.

**Worked example 1 — Bernoulli.** `n` i.i.d. coin flips `x_1, \dots, x_n \in \{0, 1\}`, model `\text{Bernoulli}(p)`. Likelihood:

$$
L(p) = \prod_i p^{x_i} (1 - p)^{1 - x_i} = p^k (1 - p)^{n - k} \quad \text{where } k = \sum_i x_i
$$

Log-likelihood:

$$
\ell(p) = k \log p + (n - k) \log(1 - p)
$$

Differentiate and set to zero:

$$
\frac{d\ell}{dp} = \frac{k}{p} - \frac{n - k}{1 - p} = 0 \quad \Longrightarrow \quad \hat{p}_{\text{MLE}} = \frac{k}{n}
$$

The MLE of a coin's bias is exactly the **sample proportion of heads** — the intuitive answer, derived rigorously.

**Worked example 2 — Gaussian.** Data from `\mathcal{N}(\mu, \sigma^2)`. We already saw the results in notebook 5: `\hat{\mu} = \bar{x}` and `\hat{\sigma}^2 = (1/n) \sum (x_i - \bar{x})^2`. The derivation is two single-variable optimizations (just two more steps than Bernoulli).

**Worked example 3 — Linear regression.** Data `(x_i, y_i)`, model `y_i = \mathbf{w}^\top \mathbf{x}_i + \epsilon_i` with `\epsilon_i \sim \mathcal{N}(0, \sigma^2)`. Log-likelihood:

$$
\ell(\mathbf{w}) = -\frac{n}{2} \log(2 \pi \sigma^2) - \frac{1}{2 \sigma^2} \sum_i (y_i - \mathbf{w}^\top \mathbf{x}_i)^2
$$

Maximize over `w` (the first term doesn't depend on it):

$$
\hat{\mathbf{w}}_{\text{MLE}} = \arg\min_{\mathbf{w}} \sum_i (y_i - \mathbf{w}^\top \mathbf{x}_i)^2
$$

That's **least-squares regression**. The famous identity: **MSE training is MLE under Gaussian noise.** Reach for an L1 loss instead and you're doing MLE under Laplace noise. Reach for cross-entropy and you're doing MLE under Categorical/Bernoulli. *Every loss is a likelihood in disguise.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

# MLE for Bernoulli — verify the sample-proportion formula
true_p = 0.3
n = 100
data = rng.random(n) < true_p

p_mle = data.mean()
print(f"True p:   {true_p}")
print(f"p MLE:    {p_mle:.4f}  (= k/n = {data.sum()}/{n})")

# Plot the log-likelihood as a function of p
ps = np.linspace(0.01, 0.99, 200)
ll = np.array([data.sum() * np.log(p) + (n - data.sum()) * np.log(1 - p) for p in ps])
plt.figure(figsize=(8, 3))
plt.plot(ps, ll)
plt.axvline(p_mle, color='r', linestyle='--', label=f"MLE = {p_mle:.3f}")
plt.axvline(true_p, color='g', linestyle=':', label=f"true = {true_p}")
plt.xlabel("p"); plt.ylabel("log-likelihood")
plt.title("Bernoulli log-likelihood")
plt.legend(); plt.show()


## Properties of MLE (worth knowing)

MLE has nice asymptotic properties — the reason it's the workhorse of frequentist statistics:

- **Consistent.** `\hat{\theta}_{\text{MLE}} \to \theta^*` (true value) as `n \to \infty`.
- **Asymptotically normal.** `\sqrt{n} (\hat{\theta}_{\text{MLE}} - \theta^*) \to \mathcal{N}(0, I(\theta^*)^{-1})`, where `I(\theta)` is the Fisher information. This is the **Cramér-Rao bound** — no unbiased estimator can do better asymptotically. MLE is asymptotically efficient.
- **Invariance.** If `\hat{\theta}` is the MLE of `\theta`, then `g(\hat{\theta})` is the MLE of `g(\theta)` for any function `g`.

**Failure modes worth knowing.**

- **Overfits with small data.** MLE picks the parameters that exactly fit what you saw. With few examples, this is often a bad estimate. Bayesian methods or regularization mitigate this.
- **Can be unbounded.** For Gaussian mixtures, the MLE can place a Gaussian on a single data point with zero variance and infinite likelihood. Pathological but real — VAE training has variations of this issue.
- **Plateaus and saddles.** Non-convex log-likelihoods (neural nets, mixtures) have many local optima and saddle points. MLE finds one of them — not necessarily the best.

**MLE in practice for ML.** Almost every neural network's training objective is negative log-likelihood, minimized by SGD. So you've been doing MLE all along, every time you trained a model.


## Bayesian inference — treating parameters as random

The frequentist view: `\theta` is a fixed but unknown number; we estimate it with `\hat{\theta}`. The Bayesian view: `\theta` is itself a random variable, with a distribution that we *update* as we see data.

**Three ingredients:**

- **Prior** `p(\theta)` — your belief about `\theta` *before* seeing the data.
- **Likelihood** `p(D | \theta)` — same as before.
- **Posterior** `p(\theta | D)` — your belief about `\theta` *after* seeing the data.

The bridge between them is **Bayes' theorem** from notebook 1:

$$
\boxed{p(\theta | D) = \frac{p(D | \theta) \, p(\theta)}{p(D)}}
$$

The denominator `p(D) = \int p(D | \theta) p(\theta) d\theta` is the **evidence** (also called marginal likelihood) — a constant in `\theta`. So we usually write proportionality:

$$
p(\theta | D) \propto p(D | \theta) \, p(\theta)
$$

**Posterior = likelihood × prior**, normalized. The shape of the posterior reflects what the data and the prior *agree* on.

**What you get from the posterior.**

- **Point estimates** — posterior mean, posterior median, posterior mode (MAP).
- **Credible intervals** — "95% credible interval" means `P(\theta \in [a, b] | D) = 0.95`. The Bayesian analogue of a confidence interval, and arguably what people *think* a confidence interval means.
- **Predictive distribution** — `p(x_{\text{new}} | D) = \int p(x_{\text{new}} | \theta) p(\theta | D) d\theta`. The full uncertainty-aware prediction, averaging over parameter uncertainty.

**Frequentist vs Bayesian — when does it matter?**

- Lots of data, well-specified model → frequentist and Bayesian agree closely.
- Little data, or strong prior knowledge available → Bayesian wins by leveraging the prior.
- Need uncertainty quantification → Bayesian gives it for free.
- Want a single number, fast → MLE/MAP gives it cheaply.


## MAP — the bridge between MLE and Bayesian inference

The **Maximum A Posteriori** estimate is the mode of the posterior:

$$
\hat{\theta}_{\text{MAP}} = \arg\max_\theta p(\theta | D) = \arg\max_\theta \bigl[p(D | \theta) p(\theta)\bigr] = \arg\max_\theta \bigl[\log p(D | \theta) + \log p(\theta)\bigr]
$$

Compare to MLE:

$$
\hat{\theta}_{\text{MLE}} = \arg\max_\theta \log p(D | \theta)
$$

**MAP = MLE + a regularizer.** The prior contributes an extra term to the objective. Whatever family of prior you pick determines what kind of regularization you get:

- **Gaussian prior** `p(\theta) = \mathcal{N}(0, \sigma^2 I)` → `\log p(\theta) = -\frac{1}{2\sigma^2} \|\theta\|^2 + \text{const}` → **L2 regularization** (ridge). The strength is `1/\sigma^2`.
- **Laplace prior** `p(\theta) = \text{Laplace}(0, b)` → `\log p(\theta) = -\frac{1}{b} \|\theta\|_1 + \text{const}` → **L1 regularization** (lasso).
- **Uniform prior** → no regularization → MAP = MLE.

So *every* regularized loss function in ML is **MAP estimation under some prior**. Weight decay in PyTorch is literally `\sigma^2 = 1/\lambda`. Dropout has a Bayesian interpretation. Early stopping is implicit regularization. The connections run deep.

**MAP vs full Bayesian.** MAP gives a single point — fast, deterministic, easy to train. Full Bayesian gives the whole posterior — more uncertainty info, but expensive and (usually) intractable, requiring MCMC or variational methods. ML overwhelmingly uses MAP for training and reserves full Bayesian methods for cases where uncertainty matters most (small data, safety-critical systems, model selection).


## Conjugate priors

When the prior and likelihood are chosen so that the posterior stays in the *same family* as the prior, the prior is called **conjugate** to the likelihood. The posterior update becomes algebraic — no integrals, no sampling.

**Conjugate pair: Beta-Bernoulli.** For a Bernoulli likelihood with parameter `p`, the Beta distribution is conjugate. If

$$
p \sim \text{Beta}(\alpha, \beta), \qquad x_i | p \sim \text{Bernoulli}(p)
$$

and you observe `k` successes in `n` trials, then

$$
p | D \sim \text{Beta}(\alpha + k, \, \beta + n - k)
$$

The posterior is also a Beta — with parameters bumped by the counts of successes and failures. Beautiful and trivially updateable as new data arrives.

**Interpretation of Beta parameters as "pseudo-counts."** `\alpha` counts "successes seen before any data," `\beta` counts "failures seen before any data." A `\text{Beta}(1, 1)` prior is uniform — no pseudo-counts, full reliance on the data. A `\text{Beta}(10, 10)` is much stronger — equivalent to seeing 10 successes and 10 failures up front.

**Conjugate pair: Gaussian-Gaussian (known variance).** For a Gaussian likelihood `x_i | \mu \sim \mathcal{N}(\mu, \sigma^2)` with known `\sigma^2`, and Gaussian prior `\mu \sim \mathcal{N}(\mu_0, \sigma_0^2)`, the posterior is also Gaussian with a precision-weighted mean. We won't write the formula here — it's a Bayesian textbook staple — but the structure is the same as Beta-Bernoulli: the parameters update algebraically.

**Why conjugacy matters.** When you can use conjugate priors, Bayesian inference is *cheaper than MLE* because the posterior is closed-form. When you can't (most real ML problems), you fall back to MCMC, variational inference, or just give up and use MAP. The exponential-family / conjugate-prior story is the bedrock of probabilistic graphical models.

**ML applications.** LDA topic models, Bayesian linear regression, Gaussian processes (Gaussian-Gaussian conjugacy in disguise), Thompson sampling for multi-armed bandits, and a lot of statistical inference packages all rely on conjugate updates.


In [ ]:
# Beta-Bernoulli: observe a coin and see how the posterior evolves

true_p = 0.3
alpha, beta = 2, 2          # prior: Beta(2, 2) — mild belief centered at 0.5

# Sample data and update sequentially
np.random.seed(0)
flips = (np.random.random(50) < true_p).astype(int)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
xs = np.linspace(0, 1, 300)

for i, n_obs in enumerate([0, 5, 20, 50]):
    k = flips[:n_obs].sum()
    a_post = alpha + k
    b_post = beta + n_obs - k
    axes[i].plot(xs, stats.beta.pdf(xs, a_post, b_post))
    axes[i].axvline(true_p, color='g', linestyle=':', label=f"true = {true_p}")
    axes[i].axvline(a_post / (a_post + b_post), color='r', linestyle='--',
                    label=f"post. mean = {a_post / (a_post + b_post):.2f}")
    axes[i].set_title(f"After {n_obs} flips ({k} heads)\nBeta({a_post}, {b_post})")
    axes[i].set_xlabel("p"); axes[i].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("Notice how the posterior tightens (less variance) and centers on the true p as data accumulates.")


## The MLE-to-loss-function bridge

Every common loss in ML corresponds to MLE under a specific noise model.

| Loss | Likelihood | Noise model |
|---|---|---|
| **MSE** for regression | Gaussian | additive Gaussian noise on `y` |
| **L1** for regression | Laplace | additive Laplace (double-exponential) noise |
| **Huber** loss | Gaussian-Laplace mix | tail-robust noise model |
| **Binary cross-entropy** | Bernoulli | binary outcome `y` modeled by `\hat{p}` |
| **Categorical cross-entropy** | Categorical | `K`-class outcome modeled by `\hat{\mathbf{p}}` |
| **Poisson NLL** | Poisson | count outcomes modeled with rate `\hat{\lambda}` |
| **Negative log-likelihood for VAE decoder** | (often Bernoulli or Gaussian) | depends on output type |

Reading this table backwards is the key insight: **the choice of loss is implicitly the choice of a probabilistic model of the noise/output.** If you change the loss, you've changed the model.

Some practical consequences:

- **Outliers.** MSE corresponds to Gaussian noise, which has *light* tails. Outliers get heavily penalized by squared error. L1 or Huber correspond to heavier-tailed noise — more robust to outliers, but slower to converge.
- **Multi-class classification.** Cross-entropy is MLE for a categorical output. There's no good non-probabilistic alternative.
- **Regression with bounded outputs.** A Beta-distributed likelihood with a sigmoid output models bounded `y \in [0, 1]` better than MSE.
- **Uncertainty.** Probabilistic losses can be replaced by their full negative log-likelihoods (predicting both mean and variance), which gives you calibrated uncertainty for free.

**Regularization.** Once you see the loss as NLL, adding L2 regularization (weight decay) is MAP under a Gaussian prior on weights. L1 (lasso) is MAP under a Laplace prior. These are not arbitrary tricks — they're proper Bayesian moves.


## Where this appears in ML

Maximum likelihood and Bayesian inference are the conceptual machinery behind almost every training procedure in ML.

- **Neural network training.** Negative log-likelihood objective optimized by SGD = MLE. Add weight decay = MAP under a Gaussian prior.
- **Logistic regression / softmax classification.** MLE under Bernoulli / Categorical with linear logits.
- **Linear regression closed form.** `\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top \mathbf{y}` is the MLE under Gaussian noise; `\hat{\mathbf{w}}_{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}` is MAP under a Gaussian prior on `\mathbf{w}`.
- **Naive Bayes.** Directly Bayesian — uses Bayes' theorem at prediction time with class-conditional likelihoods.
- **Gaussian Discriminant Analysis (LDA, QDA).** Generative MLE — fit Gaussians to each class.
- **EM algorithm.** Iteratively maximizes a lower bound on the log-likelihood for latent-variable models (GMMs, HMMs, LDA).
- **Variational inference.** Approximate the true posterior with a tractable distribution `q(\theta)`, fit by minimizing KL. Used in VAEs, Bayesian neural nets, topic models.
- **MCMC.** When you can't do conjugate updates, sample from the posterior using Metropolis-Hastings, Gibbs, HMC, or NUTS.
- **Bayesian neural networks.** Treat weights as random variables; learn approximate posteriors. Gives uncertainty estimates that point-estimate networks can't.
- **Thompson sampling.** Bayesian bandits: sample from the posterior over reward distributions to balance exploration and exploitation.
- **Bayesian optimization.** Place Gaussian process priors over objective functions to optimize expensive black-box functions with few evaluations. Used for hyperparameter tuning.
- **Probabilistic programming.** PyMC, NumPyro, Stan, Pyro, Edward — frameworks for specifying probabilistic models and doing posterior inference automatically.
- **Conformal prediction.** Distribution-free uncertainty bounds — frequentist in spirit, but Bayesian-adjacent in practice.

Next notebook: **information theory** — the conceptual home of cross-entropy, KL divergence, and the math underneath classification losses, variational inference, and self-supervised objectives.
